# ETTh1 GPU Systems Benchmark Analysis

Load MLflow runs and inspect how preprocessing backend and lookback affect speed and accuracy.

In [ ]:
import mlflow
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

mlflow.set_tracking_uri('file:../mlruns')
runs = mlflow.search_runs(experiment_names=['ts_gpu_systems_bench'])
runs.head()

In [ ]:
df = runs.copy()
for c in ['params.backend', 'params.model_type', 'params.lookback', 'metrics.val_mse', 'metrics.train_step_time_s', 'metrics.fit_in_vram']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='ignore')

df = df[df['metrics.fit_in_vram'].fillna(0) > 0.5]
df[['params.backend', 'params.model_type', 'params.lookback', 'metrics.val_mse']].head()

In [ ]:
plot_df = df.dropna(subset=['params.backend', 'params.model_type', 'params.lookback', 'metrics.val_mse']).copy()
plot_df['params.lookback'] = plot_df['params.lookback'].astype(int)
plt.figure(figsize=(10, 5))
sns.lineplot(data=plot_df, x='params.lookback', y='metrics.val_mse', hue='params.backend', style='params.model_type', markers=True, dashes=False)
plt.title('Validation MSE vs Lookback')
plt.xlabel('Lookback')
plt.ylabel('Val MSE')
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
step_df = df.dropna(subset=['params.lookback', 'metrics.train_step_time_s', 'params.backend']).copy()
step_df['params.lookback'] = step_df['params.lookback'].astype(int)
plt.figure(figsize=(10, 5))
sns.lineplot(data=step_df, x='params.lookback', y='metrics.train_step_time_s', hue='params.backend', estimator='median')
plt.title('Train Step Time vs Lookback')
plt.xlabel('Lookback')
plt.ylabel('Step Time (s)')
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
fit_df = runs.copy()
fit_df = fit_df.dropna(subset=['params.backend', 'params.model_type', 'params.lookback', 'metrics.fit_in_vram']).copy()
fit_df['params.lookback'] = pd.to_numeric(fit_df['params.lookback'], errors='coerce')
fit_df['metrics.fit_in_vram'] = pd.to_numeric(fit_df['metrics.fit_in_vram'], errors='coerce')
fit_df = fit_df[fit_df['metrics.fit_in_vram'] > 0.5]
summary = fit_df.groupby(['params.backend', 'params.model_type'])['params.lookback'].max().reset_index().rename(columns={'params.lookback': 'max_fit_lookback'})
summary.sort_values(['params.backend', 'params.model_type'])